# Frequency-Domain Features

Reduce each spectrogram frame to a handful of numbers, predict their values, and check them against librosa.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
from main import (band_energy_ratio, highpass_noise, magnitude_spectrogram, spectral_bandwidth,
                  spectral_centroid, spectral_flatness, spectral_rolloff, synthetic_vowel)

sample_rate, n_fft, hop_length = 16_000, 512, 160
sounds = {
    'vowel /a/': synthetic_vowel((730.0, 1_090.0, 2_440.0), 120.0, sample_rate, 0.5),
    '/s/-like hiss': highpass_noise(4_000.0, sample_rate, 0.5),
    'white noise': highpass_noise(0.0, sample_rate, 0.5),
}
for name, y in sounds.items():
    S = magnitude_spectrogram(y, n_fft, hop_length)[:, 5:-5]
    ber_db = 10 * np.log10(band_energy_ratio(S ** 2, 2_000.0, sample_rate, n_fft))
    print(f'{name:<14} BER {np.median(ber_db):6.1f} dB  centroid {np.median(spectral_centroid(S, sample_rate, n_fft)):6.0f} Hz  '
          f'flatness {np.median(spectral_flatness(S)):.3f}')

Before running the next cell, predict two numbers for white noise: its band energy ratio at 2 kHz (how many hertz lie on each side?) and its flatness (the lesson derives $e^{-\gamma}$).

In [ ]:
print(f'predicted BER {10 * np.log10(2_000 / 6_000):.2f} dB, predicted flatness {np.exp(-np.euler_gamma):.3f}')

In [ ]:
import librosa

y = sounds['vowel /a/'] + 0.3 * sounds['/s/-like hiss']
S = magnitude_spectrogram(y, n_fft, hop_length)
checks = {
    'centroid': (spectral_centroid(S, sample_rate, n_fft), librosa.feature.spectral_centroid(S=S, sr=sample_rate, n_fft=n_fft)[0]),
    'bandwidth': (spectral_bandwidth(S, sample_rate, n_fft), librosa.feature.spectral_bandwidth(S=S, sr=sample_rate, n_fft=n_fft)[0]),
    'rolloff': (spectral_rolloff(S, sample_rate, n_fft), librosa.feature.spectral_rolloff(S=S, sr=sample_rate, n_fft=n_fft)[0]),
    'flatness': (spectral_flatness(S), librosa.feature.spectral_flatness(S=S)[0]),
}
for name, (ours, theirs) in checks.items():
    print(f'{name:<9} max difference {np.max(np.abs(ours - theirs)):.1e}')

The features agree to floating-point precision. Now multiply `y` by 0.001 and recompute flatness for the vowel alone. Why does it change while the centroid does not? Look at how many bins fall below `amin`.